In [1]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Load & Split Data (Konsisten dengan NB 01)
df = pd.read_csv('../data/breast_cancer.csv')
df = df.drop(['id', 'Unnamed: 32'], axis=1, errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
X, y = df.drop('diagnosis', axis=1), df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Definisi Pipeline
pipelines = {
    "Dummy": Pipeline([("clf", DummyClassifier(strategy="most_frequent"))]),
    "Decision Tree": Pipeline([("clf", DecisionTreeClassifier(random_state=42))]),
    "k-Nearest Neighbor": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
    "Random Forest": Pipeline([("clf", RandomForestClassifier(random_state=42))])
}

# Ruang Hyperparameter
params = {
    "Dummy": {},
    "Decision Tree": {"clf__max_depth": [3, 5, None], "clf__min_samples_split": [2, 5]},
    "k-Nearest Neighbor": {"clf__n_neighbors": [3, 5, 7], "clf__weights": ["uniform", "distance"]},
    "Random Forest": {"clf__n_estimators": [50, 100], "clf__max_depth": [None, 5]}
}

# Eksekusi Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name in pipelines:
    grid = GridSearchCV(pipelines[name], params[name], cv=cv, scoring='f1_macro', n_jobs=-1)
    
    start_train = time.time()
    grid.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    print(f"Model: {name}")
    print(f"Best Params: {grid.best_params_}")
    print(f"F1 CV (rata-rata): {grid.best_score_:.4f}")
    print(f"Training Time: {train_time:.4f} detik\n")

Model: Dummy
Best Params: {}
F1 CV (rata-rata): 0.3851
Training Time: 9.3903 detik

Model: Decision Tree
Best Params: {'clf__max_depth': None, 'clf__min_samples_split': 2}
F1 CV (rata-rata): 0.9175
Training Time: 8.1580 detik

Model: k-Nearest Neighbor
Best Params: {'clf__n_neighbors': 3, 'clf__weights': 'uniform'}
F1 CV (rata-rata): 0.9713
Training Time: 0.6332 detik

Model: Random Forest
Best Params: {'clf__max_depth': 5, 'clf__n_estimators': 100}
F1 CV (rata-rata): 0.9646
Training Time: 3.1733 detik

